<a href="https://colab.research.google.com/github/vsujancs24-max/AI-LAB/blob/main/1WN24CS312_Week6_Propositional%20Logic_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re


def extract_symbols(kb, query):
    tokens = re.findall(r"\b[A-Z][A-Za-z0-9_]*\b", f"{kb} {query}")
    return sorted(set(tokens))


def eval_formula(formula, model):
    expr = formula
    expr = re.sub(
        r"([A-Za-z0-9_()~ ]+?)\s*<->\s*([A-Za-z0-9_()~ ]+)",
        r"((\1) == (\2))",
        expr,
    )
    expr = re.sub(
        r"([A-Za-z0-9_()~ ]+?)\s*->\s*([A-Za-z0-9_()~ ]+)",
        r"((not (\1)) or (\2))",
        expr,
    )
    expr = expr.replace("~", " not ")
    expr = expr.replace("&", " and ")
    expr = expr.replace("|", " or ")
    return bool(eval(expr, {"__builtins__": None}, model))


def tt_check_all(kb, query, symbols, model, rows):
    if not symbols:
        kb_val = eval_formula(kb, model)
        query_val = eval_formula(query, model)
        rows.append((dict(model), kb_val, query_val))
        if kb_val:
            return query_val
        return True

    first = symbols[0]
    rest = symbols[1:]

    model_false = dict(model)
    model_false[first] = False

    model_true = dict(model)
    model_true[first] = True

    return tt_check_all(kb, query, rest, model_false, rows) and tt_check_all(
        kb, query, rest, model_true, rows
    )


def tt_entails(kb, query):
    symbols = extract_symbols(kb, query)
    rows = []
    entails = tt_check_all(kb, query, symbols, {}, rows)

    header = " | ".join(f"{s:^5}" for s in symbols)
    print(f"{header} | {'KB':^5} | {'Query':^5} | {'Model of KB?'}")
    print("-" * (len(header) + 32))

    for model, kb_val, query_val in rows:
        vals = " | ".join(f"{str(model[s]):^5}" for s in symbols)
        is_model = "YES" if kb_val else "No"
        print(
            f"{vals} | {str(kb_val):^5} | {str(query_val):^5} | {is_model:^12}"
        )

    print(
        "\nResult:",
        "KB ENTAILS Query" if entails else "KB DOES NOT ENTAIL Query",
    )
    return entails


if __name__ == "__main__":
    knowledge_base = "(P -> Q) & P"
    query_to_check = "Q"

    print(f"KB: {knowledge_base}")
    print(f"Query: {query_to_check}\n")
    tt_entails(knowledge_base, query_to_check)

KB: (P -> Q) & P
Query: Q

  P   |   Q   |  KB   | Query | Model of KB?
---------------------------------------------
False | False | False | False |      No     
False | True  | False | True  |      No     
True  | False | False | False |      No     
True  | True  | True  | True  |     YES     

Result: KB ENTAILS Query
